# RAG Evaluation

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = "C:/Users/mamen/Documents/Python/RAGChatBot"

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

C:/Users/mamen/Documents/Python/RAGChatBot


In [2]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

gold_df_path = f"{PROJECT_ROOT}/eval/gold_references.csv"
gold_df = pd.read_csv(gold_df_path)

gold_df.head()

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes
0,Q001,¿Qué es la astenia?,Q001_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,9,Astenia es el término médico para denominar al cansancio (falta de energía) ya sea físico o emocional.,NaN
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,42,"Si fiebre de más de 38º C, hay que tomar un antipirético (paracetamol, ibuprofeno, metamizol… ) si tras 6-8 horas vuelve a subir la fi ebre, hay que acudir a urgencias para realizar las pruebas pertinentes.",NaN
2,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E02,general_es_pfizer_manual-pacientes_2007.pdf,34,"Si tiene temperatura de 38º C o más, debe de ir a urgencias.\nSi tiene fiebre de 38º C o superior, debe avisar a su médico o ir al hospital.",NaN
3,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,Q002_E03,general_es_pfizer_manual-pacientes_2007.pdf,150,"Una temperatura igual o superior a 38ºC indica la posibilidad de una infección y requiere consultar al médico o enfermera qué se debe hacer en una situación así. Hay ocasiones en las que algún medicamento, o la propia enfermedad, pueden provocar fiebre.",NaN
4,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,Q003_E01,general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf,47,El insomnio es el trastorno más común en los pacientes con cáncer y suele ser secundario a factores físicos y psicológicos relacionados con el cáncer y sus tratamientos.,NaN


# Validation check

In [9]:
question_id = "Q005"


gold_df[gold_df["question_id"] == question_id]

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes
6,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,Q005_E01,general_es_pfizer_manual-pacientes_2007.pdf,33,Suele durar entre 2 y 6 horas.,NaN


In [10]:
#Check which rows have the column "notes" not equal to NaN
gold_df[gold_df["notes"].notna()]

,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes


In [11]:
#Check which rows have NaN values in expected_doc, expected_page or expected_text columns
gold_df[gold_df[["expected_doc", "expected_page", "expected_text"]].isna().any(axis=1)]


,question_id,question,evidence_id,expected_doc,expected_page,expected_text,notes


In [6]:
#Prepare unique questions
questions_df = (
    gold_df[["question_id", "question"]]
    .drop_duplicates()
    .sort_values("question_id")
    .reset_index(drop=True)
)

print("Número de preguntas:", len(questions_df))
print("Número de evidencias:", len(gold_df))

Número de preguntas: 31
Número de evidencias: 77


# Evaluate Retrieval

3 types of retrieval are implemented: 
- Semantic search via function `search()`
- Hybrid search via function `hybrid_search()`: combining semantic (pgvector cosine) and keyword/full-text
- Language aware hybrid search via function `language_aware_hybrid_search()`: A wrapper around `hybrid_search` that adds a language filter to prioritize chunks in the user's language.

In [36]:
page_tolerance = 1
TOP_K = 5

## Auxiliary functions

In [37]:
def normalize_doc_name(path_or_name):
    """
    Convierte rutas tipo docs/mama/documento.pdf en documento.pdf.
    """
    if pd.isna(path_or_name):
        return ""
    return Path(str(path_or_name)).name.strip()


def normalize_page(page):
    """
    Normaliza páginas para comparar como strings.
    """
    if pd.isna(page):
        return ""
    return str(page).strip()


def get_gold_doc_page_pairs(gold_df, question_id):
    """
    Devuelve todos los pares documento-página válidos para una pregunta.
    """
    rows = gold_df[gold_df["question_id"] == question_id]

    return set(
        (
            normalize_doc_name(row["expected_doc"]),
            normalize_page(row["expected_page"])
        )
        for _, row in rows.iterrows()
    )


def retrieved_to_doc_page_pairs(retrieved_chunks):
    """
    Convierte chunks recuperados en pares documento-página.
    """
    pairs = []

    for chunk in retrieved_chunks:
        doc = normalize_doc_name(chunk.source) if chunk.source else normalize_doc_name(chunk.doc_id)
        page = normalize_page(chunk.page_num)

        pairs.append((doc, page))

    return pairs

def is_relevant_pair(retrieved_pair, gold_pairs, page_tolerance=1):

    retrieved_doc, retrieved_page = retrieved_pair

    try:
        retrieved_page = int(retrieved_page)
    except:
        return False

    for gold_doc, gold_page in gold_pairs:

        try:
            gold_page = int(gold_page)
        except:
            continue

        if (
            retrieved_doc == gold_doc
            and abs(retrieved_page - gold_page) <= page_tolerance
        ):
            return True

    return False

def is_relevant_doc(retrieved_pair, gold_pairs):
    """
    True si el documento recuperado está entre los documentos gold,
    ignorando la página.
    """
    retrieved_doc, _ = retrieved_pair

    return any(
        retrieved_doc == gold_doc
        for gold_doc, _ in gold_pairs
    )


def is_relevant_page(retrieved_pair, gold_pairs, page_tolerance=1):
    """
    True si el documento coincide y la página está dentro de ±page_tolerance.
    """
    retrieved_doc, retrieved_page = retrieved_pair

    try:
        retrieved_page = int(retrieved_page)
    except:
        return False

    for gold_doc, gold_page in gold_pairs:
        try:
            gold_page = int(gold_page)
        except:
            continue

        if retrieved_doc == gold_doc and abs(retrieved_page - gold_page) <= page_tolerance:
            return True

    return False

## Evaluate semantic search

In [38]:
from app.retrieval import search

def evaluate_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = search(question, top_k=TOP_K)
        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
        "question_id": qid,
        "question": question,
        "gold_doc_pages": gold_pairs,
        "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

        f"doc_hit@{TOP_K}": doc_hit_at_k,
        f"page_hit@{TOP_K}": page_hit_at_k,

        f"doc_precision@{TOP_K}": doc_precision_at_k,
        f"page_precision@{TOP_K}": page_precision_at_k,

        f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
        f"page_binary_recall@{TOP_K}": page_recall_at_k,

        "first_relevant_doc_rank": first_relevant_doc_rank,
        "first_relevant_page_rank": first_relevant_page_rank,

        "doc_reciprocal_rank": doc_reciprocal_rank,
        "page_reciprocal_rank": page_reciprocal_rank,
        })

    retrieval_eval_df = pd.DataFrame(retrieval_rows)

    return retrieval_eval_df

semantic_retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)
semantic_retrieval_eval_df.head()

,question_id,question,gold_doc_pages,retrieved_doc_pages_ranked,doc_hit@5,page_hit@5,doc_precision@5,page_precision@5,doc_binary_recall@5,page_binary_recall@5,first_relevant_doc_rank,first_relevant_page_rank,doc_reciprocal_rank,page_reciprocal_rank
0,Q001,¿Qué es la astenia?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 4), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 34), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 177)]",1,1,0.4,0.2,1,1,1.0,1.0,1.0,1.0
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 42), (general_es_pfizer_manual-pacientes_2007.pdf, 34), (general_es_pfizer_manual-pacientes_2007.pdf, 150)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 150), (mama_es_esmo_guia-para-pacientes_v1.pdf, 59), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 118), (general_es_pfizer_manual-pacientes_2007.pdf, 71), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 13)]",1,1,0.6,0.2,1,1,1.0,1.0,1.0,1.0
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 47)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 48), (mama_es_esmo_guia-para-pacientes_v1.pdf, 38), (prostata_es_esmo_guia-para-pacientes_2022.pdf, 41), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 152), (general_es_pfizer_manual-pacientes_2007.pdf, 72)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0
3,Q004,¿Cómo se define la mucositis?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 175), (mama_es_esmo_guia-para-pacientes_v1.pdf, 49), (mama_es_esmo_guia-para-pacientes_v1.pdf, 43)]",1,1,0.4,0.4,1,1,1.0,1.0,1.0,1.0
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,"{(general_es_pfizer_manual-pacientes_2007.pdf, 33)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 33), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 23), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 88), (mama_en_ESMO_early-breast-cancer-guidelines_2023.pdf, 7), (mama_es_esmo_guia-para-pacientes_v1.pdf, 23)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0


In [39]:
def compute_retrieval_metrics(retrieval_eval, TOP_K=5):
    """
    Calcula métricas de evaluación de recuperación a partir del DataFrame de evaluación.
    """
    retrieval_metrics = {

        # Documento
        f"Doc Hit@{TOP_K}":
            retrieval_eval[f"doc_hit@{TOP_K}"].mean(),

        f"Doc Precision@{TOP_K}":
            retrieval_eval[f"doc_precision@{TOP_K}"].mean(),

        f"Doc Binary Recall@{TOP_K}":
            retrieval_eval[f"doc_binary_recall@{TOP_K}"].mean(),

        "Doc MRR":
            retrieval_eval["doc_reciprocal_rank"].mean(),

        # Documento + Página
        f"Page Hit@{TOP_K}":
            retrieval_eval[f"page_hit@{TOP_K}"].mean(),

        f"Page Precision@{TOP_K}":
            retrieval_eval[f"page_precision@{TOP_K}"].mean(),

        f"Page Binary Recall@{TOP_K}":
            retrieval_eval[f"page_binary_recall@{TOP_K}"].mean(),

        "Page MRR":
            retrieval_eval["page_reciprocal_rank"].mean(),
    }

    retrieval_metrics_df = pd.DataFrame(
        retrieval_metrics.items(),
        columns=["metric", "value"]
    )
    return retrieval_metrics_df

semantic_retrieval_metrics_df = compute_retrieval_metrics(semantic_retrieval_eval_df, TOP_K=TOP_K)
semantic_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.367742
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.678495
4,Page Hit@5,0.645161
5,Page Precision@5,0.161290
6,Page Binary Recall@5,0.645161
7,Page MRR,0.477957


In [40]:
semantic_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_results.csv"
semantic_retrieval_eval_df.to_csv( semantic_search_eval_path,
    index=False
)
semantic_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_semantic_search_evaluation_metrics.csv", index=False)

## Evaluate hybrid search

In [41]:
from app.retrieval import hybrid_search

def evaluate_hybrid_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda híbrido para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = hybrid_search(question, top_k=TOP_K)
        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
        "question_id": qid,
        "question": question,
        "gold_doc_pages": gold_pairs,
        "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

        f"doc_hit@{TOP_K}": doc_hit_at_k,
        f"page_hit@{TOP_K}": page_hit_at_k,

        f"doc_precision@{TOP_K}": doc_precision_at_k,
        f"page_precision@{TOP_K}": page_precision_at_k,

        f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
        f"page_binary_recall@{TOP_K}": page_recall_at_k,

        "first_relevant_doc_rank": first_relevant_doc_rank,
        "first_relevant_page_rank": first_relevant_page_rank,

        "doc_reciprocal_rank": doc_reciprocal_rank,
        "page_reciprocal_rank": page_reciprocal_rank,
        })

    hybrid_retrieval_eval_df = pd.DataFrame(retrieval_rows)
    return hybrid_retrieval_eval_df

hybrid_retrieval_eval_df = evaluate_hybrid_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)

hybrid_retrieval_eval_df.head()

,question_id,question,gold_doc_pages,retrieved_doc_pages_ranked,doc_hit@5,page_hit@5,doc_precision@5,page_precision@5,doc_binary_recall@5,page_binary_recall@5,first_relevant_doc_rank,first_relevant_page_rank,doc_reciprocal_rank,page_reciprocal_rank
0,Q001,¿Qué es la astenia?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 4), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 34), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 177)]",1,1,0.4,0.2,1,1,1.0,1.0,1.0,1.0
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 42), (general_es_pfizer_manual-pacientes_2007.pdf, 34), (general_es_pfizer_manual-pacientes_2007.pdf, 150)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 150), (mama_es_esmo_guia-para-pacientes_v1.pdf, 59), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 118), (general_es_pfizer_manual-pacientes_2007.pdf, 71), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 13)]",1,1,0.6,0.2,1,1,1.0,1.0,1.0,1.0
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 47)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 48), (mama_es_esmo_guia-para-pacientes_v1.pdf, 38), (prostata_es_esmo_guia-para-pacientes_2022.pdf, 41), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 152), (general_es_pfizer_manual-pacientes_2007.pdf, 72)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0
3,Q004,¿Cómo se define la mucositis?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 175), (mama_es_esmo_guia-para-pacientes_v1.pdf, 49), (mama_es_esmo_guia-para-pacientes_v1.pdf, 43)]",1,1,0.4,0.4,1,1,1.0,1.0,1.0,1.0
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,"{(general_es_pfizer_manual-pacientes_2007.pdf, 33)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 33), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 23), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 88), (mama_en_ESMO_early-breast-cancer-guidelines_2023.pdf, 7), (mama_es_esmo_guia-para-pacientes_v1.pdf, 23)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0


In [42]:
hybrid_retrieval_metrics_df = compute_retrieval_metrics(hybrid_retrieval_eval_df, TOP_K=TOP_K)
hybrid_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.367742
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.678495
4,Page Hit@5,0.645161
5,Page Precision@5,0.161290
6,Page Binary Recall@5,0.645161
7,Page MRR,0.477957


```El sistema recuperó al menos un documento relevante para el 77.4% de las preguntas evaluadas. Cuando recuperó documentos relevantes, estos aparecieron generalmente en las primeras posiciones del ranking (MRR=0.678). Considerando además la localización precisa de la información a nivel de página (±1 página de tolerancia), la tasa de éxito fue del 64.5% (Page Hit@5), con un MRR de 0.478. Estos resultados sugieren que el sistema identifica razonablemente bien los documentos pertinentes, aunque existe margen de mejora en la recuperación del fragmento específico que contiene la respuesta.```

In [12]:
hybrid_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_hybrid_search_evaluation_results.csv"
hybrid_retrieval_eval_df.to_csv( hybrid_search_eval_path,
    index=False
)

hybrid_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_hybrid_search_evaluation_metrics.csv", index=False)

## Evaluate language aware hybrid search 

In [43]:
from app.retrieval import language_aware_hybrid_search

def evaluate_language_aware_hybrid_search(questions_df, gold_df, page_tolerance=1, TOP_K=5):
    """Evalua un motor de búsqueda híbrido sensible al lenguaje para un conjunto de preguntas y referencias gold."""

    retrieval_rows = []

    for _, row in questions_df.iterrows():
        qid = row["question_id"]
        question = row["question"]

        #Obtener los pares de documento-página de referencia para la pregunta actual
        #Devuelve un conjunto de tuplas (doc, page) donde cualquiera de esos documentos/páginas cuenta como evidencia correcta para la pregunta.
        gold_pairs = get_gold_doc_page_pairs(gold_df, qid)

        #Ejecuta el retrieval
        retrieved_chunks = language_aware_hybrid_search(question, top_k=TOP_K)
        # Convierte los chunks recuperados a pares documento-página, el orden se conserva para poder calcular métricas de ranking.
        retrieved_pairs_ranked = retrieved_to_doc_page_pairs(retrieved_chunks)

        #Compara cada documento/página recuperado contra todos los documentos/páginas válidos.
        # LImitación: si hay chunks de la misma página, se puede contar como hit varias veces. 
        doc_hits_by_rank = [
            is_relevant_doc(pair, gold_pairs)
            for pair in retrieved_pairs_ranked
        ]

        page_hits_by_rank = [
            is_relevant_page(pair, gold_pairs, page_tolerance=page_tolerance)
            for pair in retrieved_pairs_ranked
        ]
        
        # Calcular métricas de evaluación
        #Hit@K: ¿Hay al menos un resultado correcto entre los top K?
        doc_hit_at_k = int(any(doc_hits_by_rank))
        page_hit_at_k = int(any(page_hits_by_rank))

        #Identificar en qué posición apareció la primera evidencia correcta.
        first_relevant_doc_rank = None
        for i, is_hit in enumerate(doc_hits_by_rank, start=1):
            if is_hit:
                first_relevant_doc_rank = i
                break

        first_relevant_page_rank = None
        for i, is_hit in enumerate(page_hits_by_rank, start=1):
            if is_hit:
                first_relevant_page_rank = i
                break

        # Calcular el Reciprocal Rank (RR) para la pregunta actual.
        # El RR es 1 dividido por la posición de la primera evidencia correcta. Si no hay evidencia correcta, el RR es 0.
        # Esta métrica premia que la evidencia correcta aparezca arriba.
        doc_reciprocal_rank = (
            0 if first_relevant_doc_rank is None else 1 / first_relevant_doc_rank
        )

        page_reciprocal_rank = (
            0 if first_relevant_page_rank is None else 1 / first_relevant_page_rank
        )

        #Cuenta cuántas evidencias correctas recuperó
        num_relevant_docs_retrieved = sum(doc_hits_by_rank)
        num_relevant_pages_retrieved = sum(page_hits_by_rank)

        #El recall_at_k es básicamente igual que hit_at_k porque estamos usando recall binario.
        # Recall@K: ¿Se recuperó al menos una evidencia correcta entre los top K?
        doc_recall_at_k = int(num_relevant_docs_retrieved > 0)  # binario
        page_recall_at_k = int(num_relevant_pages_retrieved > 0)

        #Precision@K: ¿Qué proporción de los resultados recuperados son correctos? De los K chunks recuperados, ¿cuántos eran documento/página válidos?
        doc_precision_at_k = num_relevant_docs_retrieved / TOP_K
        page_precision_at_k = num_relevant_pages_retrieved / TOP_K

        retrieval_rows.append({
        "question_id": qid,
        "question": question,
        "gold_doc_pages": gold_pairs,
        "retrieved_doc_pages_ranked": retrieved_pairs_ranked,

        f"doc_hit@{TOP_K}": doc_hit_at_k,
        f"page_hit@{TOP_K}": page_hit_at_k,

        f"doc_precision@{TOP_K}": doc_precision_at_k,
        f"page_precision@{TOP_K}": page_precision_at_k,

        f"doc_binary_recall@{TOP_K}": doc_recall_at_k,
        f"page_binary_recall@{TOP_K}": page_recall_at_k,

        "first_relevant_doc_rank": first_relevant_doc_rank,
        "first_relevant_page_rank": first_relevant_page_rank,

        "doc_reciprocal_rank": doc_reciprocal_rank,
        "page_reciprocal_rank": page_reciprocal_rank,
        })

    lang_retrieval_eval_df = pd.DataFrame(retrieval_rows)
    return lang_retrieval_eval_df

lang_retrieval_eval_df = evaluate_language_aware_hybrid_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=TOP_K)
lang_retrieval_eval_df.head()

,question_id,question,gold_doc_pages,retrieved_doc_pages_ranked,doc_hit@5,page_hit@5,doc_precision@5,page_precision@5,doc_binary_recall@5,page_binary_recall@5,first_relevant_doc_rank,first_relevant_page_rank,doc_reciprocal_rank,page_reciprocal_rank
0,Q001,¿Qué es la astenia?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 9), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 4), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 34), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 177)]",1,1,0.4,0.2,1,1,1.0,1.0,1.0,1.0
1,Q002,¿Qué debe hacer un paciente si tiene una temperatura de 38º o más?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 42), (general_es_pfizer_manual-pacientes_2007.pdf, 34), (general_es_pfizer_manual-pacientes_2007.pdf, 150)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 150), (mama_es_esmo_guia-para-pacientes_v1.pdf, 59), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 118), (general_es_pfizer_manual-pacientes_2007.pdf, 71), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 13)]",1,1,0.6,0.2,1,1,1.0,1.0,1.0,1.0
2,Q003,¿Cuál es el trastorno del sueño más común en pacientes con cáncer?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 47)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 48), (mama_es_esmo_guia-para-pacientes_v1.pdf, 38), (prostata_es_esmo_guia-para-pacientes_2022.pdf, 41), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 152), (general_es_pfizer_manual-pacientes_2007.pdf, 72)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0
3,Q004,¿Cómo se define la mucositis?,"{(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14)}","[(general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (general_es_gepac_guia-toxicidad-quimioterapia_v1.pdf, 14), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 175), (mama_es_esmo_guia-para-pacientes_v1.pdf, 49), (mama_es_esmo_guia-para-pacientes_v1.pdf, 43)]",1,1,0.4,0.4,1,1,1.0,1.0,1.0,1.0
4,Q005,¿Cuánto tiempo suele durar una sesión de administración de quimioterapia?,"{(general_es_pfizer_manual-pacientes_2007.pdf, 33)}","[(general_es_pfizer_manual-pacientes_2007.pdf, 33), (mama_es_novartis_guia-pacientes-CM_2025.pdf, 23), (mama_es_HUReinaSofia_protocolo-cancer-mama_2021.pdf, 88), (mama_es_esmo_guia-para-pacientes_v1.pdf, 23), (prostata_es_gepac_guia-cancer-de-prostata_2020.pdf, 36)]",1,1,0.2,0.2,1,1,1.0,1.0,1.0,1.0


In [44]:
lang_retrieval_metrics_df = compute_retrieval_metrics(lang_retrieval_eval_df, TOP_K=TOP_K)

lang_retrieval_metrics_df

,metric,value
0,Doc Hit@5,0.774194
1,Doc Precision@5,0.387097
2,Doc Binary Recall@5,0.774194
3,Doc MRR,0.654301
4,Page Hit@5,0.645161
5,Page Precision@5,0.174194
6,Page Binary Recall@5,0.645161
7,Page MRR,0.507527


In [45]:
lang_search_eval_path = f"{PROJECT_ROOT}/eval/retrieval_language_aware_hybrid_search_evaluation_results.csv"
lang_retrieval_eval_df.to_csv( lang_search_eval_path,
    index=False
)

lang_retrieval_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_language_aware_hybrid_search_evaluation_metrics.csv", index=False)

## Combine metrics in a single dataFrame

In [46]:
#Combine retrieval_metrics_df, hybrid_retrieval_metrics_df, lang_retrieval_metrics_df into a single dataframe with an additional column "retrieval_method" indicating the method used.
#The rows will be only 8 of the 8 metrics computed and the values of each method will be added to a column named by the retrieval methdd "semantic_search", "hybrid_search", "language_aware_hybrid_search".

combined_metrics_df = pd.DataFrame({
    "metric": semantic_retrieval_metrics_df["metric"],
    "semantic_search": semantic_retrieval_metrics_df["value"],
    "hybrid_search": hybrid_retrieval_metrics_df["value"],
    "language_aware_hybrid_search": lang_retrieval_metrics_df["value"]
})

combined_metrics_df.to_csv( f"{PROJECT_ROOT}/eval/retrieval_combined_evaluation_metrics.csv", index=False)
combined_metrics_df

,metric,semantic_search,hybrid_search,language_aware_hybrid_search
0,Doc Hit@5,0.774194,0.774194,0.774194
1,Doc Precision@5,0.367742,0.367742,0.387097
2,Doc Binary Recall@5,0.774194,0.774194,0.774194
3,Doc MRR,0.678495,0.678495,0.654301
4,Page Hit@5,0.645161,0.645161,0.645161
5,Page Precision@5,0.161290,0.161290,0.174194
6,Page Binary Recall@5,0.645161,0.645161,0.645161
7,Page MRR,0.477957,0.477957,0.507527


```
There is no apparent meaningful difference in retrieval coverage between the 3 methods. Language aware and hybrid search may be mostly the same because all the questions are in spanish and most of the documents are also in spanish and that the relevant English documents may not be needed (or as relevant) to answer the benchmark questions.

Equal Doc and Page Hit@5 indicate that the same questions are being solved and missed by all three systems. But Language Aware Hybrid slightly improves precision.

Doc MRR for semantic search and hybrid is higher than language aware search, which could mean that, when looking only at documents, semantic and hybrid search places the first correct document slightly higher in the ranking.

Page MRR for language aware search is higher, which could mean that language aware retrieval finds the correct page slightly earlier than semantic search.
```

De forma orientativa, valores aceptables de retrieval son: 

| Métrica          | Malo | Aceptable | Bueno   | Muy bueno |
| ---------------- | ---- | --------- | ------- | --------- |
| Recall@5 / Hit@5 | <60% | 60-80%    | 80-90%  | >90%      |
| MRR              | <0.4 | 0.4-0.6   | 0.6-0.8 | >0.8      |

Ahora mismo los resultados son buenos/aceptables pero podrían mejorarse más

Opciones para mejorar las métricas:
- Evaluar con mayor tolearancia page_tolearance (2 o 3)
- Mejorar el chunking: 
    - Probar chunks más pequeños (400-500 tokens) (no por página)
    - Mayor overlap (100-150 tokens)
    - Chunking semántico separando por encabezados, párrafos o secciones
- Probar a aumentar el TOP-10 recalcular Hit@10 y MRR@10. Si el Hit sube mucho, el retrieval encuentra información pero demsiado abajo en el ranking.
- Query expansion
- Reranker
- Revisar el modelo de embedding

# Investigate metrics for different TOP_K and page_tolerance values (semantic)

In [17]:
TOP_K = 5
page_tolerance =1

In [ ]:
page_tolerance_values = [0, 1, 2, 3]

combined_tolerance_semantic_metrics_df = pd.DataFrame({
    "metric": semantic_retrieval_metrics_df["metric"],
    "page_tolerance_0": None,
    "page_tolerance_1": None,
    "page_tolerance_2": None,
    "page_tolerance_3": None
})

#Evaluate and compare differences in metrics for different page_tolerance values. For each page_tolerance value, run the evaluate_search function and store the results in a dictionary with the page_tolerance as the key and the resulting dataframe as the value.
for page_tol in page_tolerance_values:
    retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tol, TOP_K=TOP_K)
    retrieval_metrics_df = compute_retrieval_metrics(retrieval_eval_df, TOP_K=TOP_K)
    combined_tolerance_semantic_metrics_df[f"page_tolerance_{page_tol}"] = retrieval_metrics_df["value"]

combined_tolerance_semantic_metrics_df
    

,metric,page_tolerance_0,page_tolerance_1,page_tolerance_2,page_tolerance_3
0,Doc Hit@5,0.774194,0.774194,0.774194,0.774194
1,Doc Precision@5,0.367742,0.367742,0.367742,0.367742
2,Doc Binary Recall@5,0.774194,0.774194,0.774194,0.774194
3,Doc MRR,0.678495,0.678495,0.678495,0.678495
4,Page Hit@5,0.580645,0.645161,0.677419,0.709677
5,Page Precision@5,0.135484,0.161290,0.200000,0.225806
6,Page Binary Recall@5,0.580645,0.645161,0.677419,0.709677
7,Page MRR,0.423118,0.477957,0.494086,0.526344


In [24]:
page_tolerance = 1
top_k_values = [1, 3, 5, 10]
metrics_list = ["Doc Hit@K", "Doc Precision@K", "Doc Binary Recall@K", "Doc MRR", "Page Hit@K", "Page Precision@K", "Page Binary Recall@K", "Page MRR"]

combined_tok_k_semantic_metrics_df = pd.DataFrame({
    "metric": metrics_list,
    "top_k_1": None,
    "top_k_3": None,
    "top_k_5": None,
    "top_k_10": None
})

for top_k in top_k_values:
    retrieval_eval_df = evaluate_search(questions_df, gold_df, page_tolerance=page_tolerance, TOP_K=top_k)
    retrieval_metrics_df = compute_retrieval_metrics(retrieval_eval_df, TOP_K=top_k)
    combined_tok_k_semantic_metrics_df[f"top_k_{top_k}"] = retrieval_metrics_df["value"]

combined_tok_k_semantic_metrics_df

,metric,top_k_1,top_k_3,top_k_5,top_k_10
0,Doc Hit@K,0.612903,0.741935,0.774194,0.870968
1,Doc Precision@K,0.612903,0.430108,0.367742,0.338710
2,Doc Binary Recall@K,0.612903,0.741935,0.774194,0.870968
3,Doc MRR,0.612903,0.672043,0.678495,0.690937
4,Page Hit@K,0.387097,0.548387,0.645161,0.741935
5,Page Precision@K,0.387097,0.215054,0.161290,0.122581
6,Page Binary Recall@K,0.387097,0.548387,0.645161,0.741935
7,Page MRR,0.387097,0.456989,0.477957,0.489823


```
Big difference between Doc and Page Hit = The system often finds the correct document, but not the correct location within that document.
The retrieval is finding relevant information but is not ranking it optimally. 

The large increase from K=1 to K=10 means that relevant information often exists among the retrieved candidates, but it is not appearing at the top K of the ranking.
Difference in Doc and Page MRR = The correct document tends to appear near the top, but the correct chunk/page is often ranked below another chunk from the same document. 
This can mean that the ranking is a bigger problem than the retrieval. If retrieval itself were the main problem, Hit@10 would remain low. 

Would be interesting to analyze failures where page_hit@10 = 0. 

Conclusion 1: top-5 may be unnecessarily restrictive. 
Conclusion 2: A reranker may help more than any other fix. Also improving chunking could help.
Conclusion 3: The retrieval corpus is quite good, the problem may not be the retrieval. 


The decrease in precision is normal. When K increases, recall and hit increase while precision decreases. This is an expected tradeoff. 
```

# Content evaluation (TODO)

In [35]:
#Generate the answers using RAG 
from huggingface_hub import InferenceClient
from app.config import settings
from app.rag import rag_answer

TOP_K = 5

hf_client = InferenceClient(
    provider="novita",
    api_key=settings.hf_api_key
)

answer_rows = []

for _, row in questions_df.iterrows():
    qid = row["question_id"]
    question = row["question"]

    answer = rag_answer(
        hf_client=hf_client,
        user_message=question,
        chat_history=[],
        model=settings.llm_model_name,
        top_k=TOP_K,
    )

    answer_rows.append({
        "question_id": qid,
        "question": question,
        "answer": answer,
    })

answers_df = pd.DataFrame(answer_rows)

answers_df.head()

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/novita/v3/openai/chat/completions (Request ID: Root=1-6a3bd26a-61638d48349d937613845670;aa447773-156a-4c30-b3ea-8f15752efc44)

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

In [ ]:
#Save the generated answers to a CSV file in the eval folder.

from pathlib import Path
import pandas as pd

OUT_PATH = f"{PROJECT_ROOT} /eval/ generated_answers_partial.csv"
output_path = Path(OUT_PATH)
existing = pd.read_csv(output_path) if output_path.exists() else pd.DataFrame()
done_ids = set(existing["question_id"]) if not existing.empty else set()

rows = []

for _, row in questions_df.iterrows():
    qid = row["question_id"]

    if qid in done_ids:
        continue

    question = row["question"]

    try:
        answer = rag_answer(
            hf_client=hf_client,
            user_message=question,
            chat_history=[],
            model=settings.llm_model_name,
            top_k=5,
        )

        new_row = pd.DataFrame([{
            "question_id": qid,
            "question": question,
            "answer": answer,
        }])

        new_row.to_csv(
            output_path,
            mode="a",
            header=not output_path.exists(),
            index=False,
        )

    except Exception as e:
        print(f"Stopped at {qid}: {e}")
        break

In [ ]:
generated_answers_path = f"{PROJECT_ROOT}/eval/generated_answers.csv"
answers_df.to_csv(
    generated_answers_path,
    index=False
)

NameError: name 'answers_df' is not defined

## Crear CSV para evaluación manual de completeness y faithfulness

completeness:
- 1 = la respuesta incluye la información esperada
- 0 = la respuesta no incluye la información esperada

faithfulness:
- 1 = la respuesta está respaldada por los documentos/evidencias
- 0 = añade información no soportada o contradice las evidencias

In [ ]:
content_eval_df = answers_df.merge(
    semantic_retrieval_eval_df[
        [
            "question_id",
            f"hit@{TOP_K}",
            f"precision@{TOP_K}",
            f"binary_recall@{TOP_K}",
            "first_relevant_rank",
            "reciprocal_rank",
            "retrieved_doc_pages_ranked",
            "gold_doc_pages",
        ]
    ],
    on="question_id",
    how="left"
)

content_eval_df["completeness"] = ""
content_eval_df["faithfulness"] = ""
content_eval_df["notes"] = ""

content_evaluation_manual_path = f"{PROJECT_ROOT}/eval/content_evaluation_manual.csv"
content_eval_df.to_csv(
    content_evaluation_manual_path,
    index=False
)

content_eval_df.head()

## Calcular métricas de contenido después de rellenar el CSV

In [ ]:

manual_eval_df = pd.read_csv(content_evaluation_manual_path)

manual_eval_df["completeness"] = pd.to_numeric(
    manual_eval_df["completeness"],
    errors="coerce"
)

manual_eval_df["faithfulness"] = pd.to_numeric(
    manual_eval_df["faithfulness"],
    errors="coerce"
)

content_metrics = {
    "Completeness": manual_eval_df["completeness"].mean(),
    "Faithfulness": manual_eval_df["faithfulness"].mean(),
    "Retrieval Hit": manual_eval_df[f"hit@{TOP_K}"].mean(),
}

content_metrics_df = pd.DataFrame(
    content_metrics.items(),
    columns=["metric", "value"]
)

content_metrics_df

In [ ]:
for metric, value in content_metrics.items():
    print(f"{metric}: {value:.2%}")

# Metrica combinada: porcentaje de preguntas donde retrieval acertó, la respuesta fue completa y además fiel.

In [ ]:
manual_eval_df["end_to_end_success"] = (
    (manual_eval_df[f"hit@{TOP_K}"] == 1)
    & (manual_eval_df["completeness"] == 1)
    & (manual_eval_df["faithfulness"] == 1)
).astype(int)

end_to_end_score = manual_eval_df["end_to_end_success"].mean()

print(f"End-to-end success: {end_to_end_score:.2%}")

# Resumen de metricas

In [ ]:
final_metrics = {
    f"Hit@{TOP_K}": manual_eval_df[f"hit@{TOP_K}"].mean(),
    f"Precision@{TOP_K}": manual_eval_df[f"precision@{TOP_K}"].mean(),
    f"Binary Recall@{TOP_K}": manual_eval_df[f"binary_recall@{TOP_K}"].mean(),
    "MRR": manual_eval_df["reciprocal_rank"].mean(),
    "Completeness": manual_eval_df["completeness"].mean(),
    "Faithfulness": manual_eval_df["faithfulness"].mean(),
    "End-to-end success": manual_eval_df["end_to_end_success"].mean(),
}

final_metrics_df = pd.DataFrame(
    final_metrics.items(),
    columns=["metric", "value"]
)

final_metrics_df

In [ ]:
final_metrics_df.to_csv(
    PROJECT_ROOT / "rag_evaluation_metrics_summary.csv",
    index=False
)

manual_eval_df.to_csv(
    PROJECT_ROOT / "rag_evaluation_full_results.csv",
    index=False
)